# Домашнее задание: доменная адаптация LLM

В рамках этого ноутбука мы попробуем решить довольно популярную сейчас в российских IT-компаниях задачу, правда, в более маленьком масштабе. Мы затюним китайскую модель под русский язык.
Что охватим:
1. Замеры бенчмарков сторонними библиотеками
2. Токенизация и построение датасета
3. Обучение LoRA модели
4. Инференс

# 0. Important imports

Для начала импортируем нужные штуки. Будем использовать библиотеку от https://unsloth.ai. Они написали очень оптимальный код для дообучения LLM под свои задачи.

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

# 1. Base model — 10 баллов

В качестве базовой модели возьмем Qwen2.5-1.5B — это достаточно маленькая модель, она спокойно и быстро обучится в Google Colab. Давайте скачаем ее локально в виде директории, чтобы иметь к ней быстрый доступ. Будем скачивать ее из репозитория https://huggingface.co/unsloth/Qwen2.5-1.5B. Убедитесь, что у вас установлен `git lfs`.

In [ ]:
# https://github.com/git-lfs/git-lfs/blob/main/INSTALLING.md
!git clone https://huggingface.co/unsloth/Qwen2.5-1.5B

### FastLanguageModel — 3 балла
Обернем нашу модель в класс `FastLanguageModel` из unsloth, документацию можно найти [тут](https://docs.unsloth.ai/get-started/fine-tuning-guide#id-2.-choose-the-right-model--method). max_seq_length установим в 2048 и загрузим модель в 4 битах для ускорения обучения.

In [ ]:
from unsloth import FastLanguageModel
import torch
# ---- Ваш код здесь ----
# ---- Конец кода ----

model, tokenizer = FastLanguageModel.from_pretrained(
    # ---- Ваш код здесь ----
    # ---- Конец кода ----
)

### Генерируем ответ базовой моделью — 4 балла

Теперь давайте попробуем оценить возможности нашей модели прямо из коробки. Для этого напишем функцию `generate_answer`, которая будет принимать на вход промпт и возвращать нам ответ. Несмотря на то, что наша модель является базовой, то есть не обученной под диалоговый формат, мы все равно предполагаем, что диалог она понимает. Оформите функцию так, чтобы можно было добавить опционально шаблон диалога и сравнить результаты с ним и без него:

In [ ]:
from transformers import AutoTokenizer
FastLanguageModel.for_inference(model)
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B")

def generate_answer(prompt, apply_chat_template=False):
    # ---- Ваш код здесь ----
    raise NotImplemented()
    # ---- Конец кода ----

print(generate_answer("Расскажи мне про Булгакова"))

### Оцениваем возможности и собираем корзинку — 3 балла

Запишите свои выводы в ячейку с markdown ниже. Меняется ли поведение модели при добавлении диалога? Попробуйте промпты на русском, английском и (на всякий случай) китайском.

**Выводы:**
\# TODO: выводы о поведении модели

Придумайте "корзинку" из 10 диалоговых промптов на русском языке. Выведите ответы модели на этой корзинке, по ней будем оценивать результат обучения. Можете придумать сами или взять интересные примеры отсюда: https://llmarena.ru/ -> Arena Explorer. Учтите, что проверять стоит скорее диалоговость, а не знания модели.

In [ ]:
# ---- Ваш код здесь ----
# ---- Конец кода ----

# 2. Бенчмарки — 10 баллов

Давайте замерим нашу модель с помощью бенчмарка LM Evaluation Harness — документацию можно найти [тут](https://github.com/EleutherAI/lm-evaluation-harness), установить его можно по инструкции из репозитория. Это нужно будет, чтобы убедиться, что мы не просадили качество нашим обучением. Запускать бенчмарк будем из командной строки.

In [ ]:
!git clone --depth 1 https://github.com/EleutherAI/lm-evaluation-harness
!cd lm-evaluation-harness && pip install -e .

Если будет падать с ошибкой `HTTP Error 429`, то придется залогиниться в HF. Это особенность колаба

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

Напишите bash скрипт для прогона бенчмарков:

In [ ]:
%%writefile run_lmeh.sh
# ---- Ваш код здесь ----
# ---- Конец кода ----

In [ ]:
!bash run_lmeh.sh

В качестве бенчмарков выберете `truthfulqa_ru_mc1` и еще 2 любых на ваше усмотрение — главное, чтобы они быстро пробежали (не более 10 минут). В ответ напишите 3 строчки с результатами бенчмарков.

**Результаты замеров:** \# TODO: Results


# 3. Готовим данные — 5 баллов

Довольно часто возникает задача адаптации базовой модели под русский язык. В рамках этого ноутбука хочется обучиться на датасете с русскоязычными инструкциями.
Для дообучения модели будем использовать датасет `Vikhrmodels/GrandMaster-PRO-MAX`. Было решено остановиться именно на нем, так как он самый свежий и модели, обученные на этом датасете, показывают хорошее качество.

Однако на HuggingFace много и других инструктивных датасетов под русский язык, например тут: https://huggingface.co/collections/IlyaGusev/

Очень часто вы будете сталкиваться с разными шаблонами диалога, как-то модифицировать его или даже готовить собственный. Давайте в рамках подготовки датасета для обучения создадим несложный шаблон вида:

```text
<|im_start|>user
Hi there!<|im_end|>
<|im_start|>assistant
Nice to meet you!<|im_end|>
<|im_start|>user
Can I ask a question?<|im_end|>
```

Про шаблоны можно почитать тут:
https://huggingface.co/docs/transformers/en/chat_templating

In [ ]:
def formatting_func(example):
    # ---- Ваш код здесь ----
    raise NotImplemented()
    # ---- Конец кода ----

In [ ]:
chat = { "conversation": [
  {"role": "user", "content": "Hello!"},
  {"role": "assistant", "content": "How can I help you today?"},
  {"role": "user", "content": "I dont know!"},
]}

assert formatting_func(chat)["text"] == '<|im_start|>user\nHello!<|im_end|>\n<|im_start|>assistant\nHow can I help you today?<|im_end|>\n<|im_start|>user\nI dont know!<|im_end|>\n'

Шаблонизируем наш датасет:

In [ ]:
from datasets import load_dataset
vikhr_dataset = load_dataset("Vikhrmodels/GrandMaster-PRO-MAX", split = "train")
vikhr_dataset = vikhr_dataset.map(formatting_func, batched = False,)

# 4. Обучаем нашу модель — 10 баллов

### Конфигурация LoRA — 5 баллов

Мы будем проводить обучение на инструктивный датасет с помощью библиотеки `TRL SFT`, ее документация [тут](https://huggingface.co/docs/trl/sft_trainer). Ваша задача — подобрать конфигурацию для обучения. Обучение должно уложиться в 1 час на GPU T4. В качестве `target_modules` возьмите все матрицы из подблока attention и mlp, их должно быть 7. Остальные гиперпараметры можно варьировать.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    # ---- Ваш код здесь ----
    # ---- Конец кода ----
)

### Обучение модели — 5 баллов

In [ ]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = vikhr_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    # ---- Ваш код здесь ----
    # ---- Конец кода ----
)

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

Запускаем обучение:

In [ ]:
trainer_stats = trainer.train()

На всякий случай сохраним прогресс:

In [ ]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

In [ ]:
!tar -czvf lora_model.tar.gz lora_model

Обязательно скачайте модель локально, иначе вы утратите прогресс при сбросе колаба!

# 5. Инференс модели — 15 баллов



### Шаблон токенизации — 5 баллов

Для начала нам нужно внести шаблон диалога в наш токенизатор, чтобы его было удобно использовать. Для этого нужно модифицировать `tokenizer.chat_template`

In [ ]:
# ---- Ваш код здесь ----
# ---- Конец кода ----

Проверим, что мы все сделали верно:

In [ ]:
chat = [
  {"role": "user", "content": "Hello!"},
  {"role": "assistant", "content": "How can I help you today?"},
  {"role": "user", "content": "I dont know!"},
]

assert tokenizer.apply_chat_template(chat, tokenize=False) == '<|im_start|>user\nHello!<|im_end|>\n<|im_start|>assistant\nHow can I help you today?<|im_end|>\n<|im_start|>user\nI dont know!<|im_end|>\n'

Теперь давайте вмержим наш LoRA адаптер в модель, чтобы хранить ее целиком. Сохраним в формате `gguf`, это понадобится нам для инференса в дальнейшем.

In [ ]:
model.save_pretrained_gguf("qwen25_15_ru_instruct", tokenizer,)

Если у вас заканчивается оперативная память, то на этом моменте вы можете перезагрузить среду выполнения. Главное — сохранить прогресс на локальный комптютер. В следующей ячейке пример как можно восстановить чекпоинт:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    import torch
    # ---- Ваш код здесь ----
    # ---- Конец кода ----
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model",
        # ---- Ваш код здесь ----
        # ---- Конец кода ----
    )

Проверим хотя бы на 1 примере что теперь мы понимаем русский язык:

In [ ]:
FastLanguageModel.for_inference(model)
inputs = tokenizer.apply_chat_template([
    {"role": "user", "content": "<Ваш пример тут>"},
], return_tensors = "pt", add_generation_prompt=True).to("cuda")

In [ ]:
outputs = model.generate(inputs, max_new_tokens = 64, use_cache = True)
tokenizer.decode(outputs[0])

Теперь снова замерим нашу модель на бенчмарках, чтобы убедиться, что мы ничего не ухудшили:

In [ ]:
%%writefile run_lmeh.sh
    # ---- Ваш код здесь ----
    # ---- Конец кода ----

### Инференс llama.cpp — 5 баллов
Теперь давайте запустим инференс нашей модели через `llama.cpp`. Это один из самых популярных фреймворков для инференса языковых моделей локально. Его документацию можно почитать [тут](https://github.com/ggml-org/llama.cpp/tree/master?tab=readme-ov-file#quick-start). Запустить его нужно в режиме сервера через терминал в Google Colab (внизу в футтере). А из ноутбука с помощью OpenAI API прокачать:

In [ ]:
!git clone https://github.com/ggml-org/llama.cpp
!cd llama.cpp && cmake -B build && cmake --build build --config Release

In [ ]:
model.save_pretrained_gguf("qwen25_15_ru_instruct", tokenizer,)

In [ ]:
import openai

client = openai.Client(
    # ---- Ваш код здесь ----
    # ---- Конец кода ----
)

response = client.chat.completions.create(
    # ---- Ваш код здесь ----
    # ---- Конец кода ----
)

In [ ]:
response.choices[0].message.content

### Прогон на корзинке — 5 баллов

Прогоним корзинку из самого начала и убедимся что наша модель стала хорошо отвечать на диалоги:

In [ ]:
# ---- Ваш код здесь ----
# ---- Конец кода ----